# 03 – Model Experiments

Compare TF-IDF + Logistic Regression vs TF-IDF + Linear SVM for fit classification.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = Path('../data/processed')

In [ ]:
training_path = PROCESSED_DIR / 'training_fit_dataset.csv'
if training_path.exists():
    df = pd.read_csv(training_path)
    df['text'] = df['title'].fillna('') + ' ' + df['student_major'].fillna('')
    X = df['text']
    y = df['fit_label']
    print(f'Dataset: {len(df)} rows | Label distribution:')
    print(y.value_counts())
else:
    print('Run build_training_data.py first.')

In [ ]:
if 'X' in dir():
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec  = vectorizer.transform(X_test)

    models = {
        'Logistic Regression': LogisticRegression(max_iter=500, class_weight='balanced'),
        'Linear SVM': LinearSVC(class_weight='balanced'),
    }

    for name, clf in models.items():
        clf.fit(X_train_vec, y_train)
        y_pred = clf.predict(X_test_vec)
        print(f'\n=== {name} ===')
        print(classification_report(y_test, y_pred))